In [2]:
import pandas as pd

In [9]:
%pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.0 kB/s eta 0:00:00a 0:00:07
Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install stop_words

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for stop_words: filename=stop_words-2018.7.23-py3-none-any.whl size=32988 sha256=67a99cb888566814b4aead7a7dcfae295e0344dfa0e2c8d63d2aa67ee26e988a
  Stored in directory: /Users/vera/Library/Caches/pip/wheels/98/8d/87/5894deb0270ab49fc65555daa606a7d1dfa144f456bb9e0795
Successfully built stop_words
Note: you may need to restart the kernel to use updated packages.


In [13]:
%pip install natasha

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 181.8 kB/s eta 0:00:0000:0100:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 115.5 kB/s eta 0:00:0000:0100:03
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=86c825fe1cb887302d7a459c3b8249e79bad9560928f1a03c9304acc5ee3028f
  Stored in directory: /Users/vera/Library/Caches/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
  Created wheel for intervaltree: filename=intervaltree-3.1.0-py2.py3-none-any.whl size=26189 sha256=899ddbad5288e4002b91c4d94b15a1a4429da08f8df29fa08fd92c06b783e5e4
  Stored in directory: /Users/vera/Library/Caches/pip/wheels/65/c3/c3/238bf93c243597857edd94ddb0577faa74a8e16e9

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.cluster.util import cosine_distance
import numpy as np
import networkx as nx
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
import pandas as pd
from nltk.tokenize import word_tokenize
import string
import math

def read_article(text):
    filedata = text
    report = filedata.split(". ")
    sentences = []
    lm = WordNetLemmatizer()
    for sentence in report:
        sentence = sentence.replace("\n", " ")
        sentence = sentence.replace("[^а-яА-Я]", " ")
        sentence = [lm.lemmatize(w.lower().strip(string.punctuation)) for w in sentence.split(" ") if w.lower().strip(string.punctuation) != '']
        if len(sentence) > 0:
            sentences.append(sentence)
    sentences.pop()
    
    return sentences

def tfidf(report, word):
    count = 0
    for sentence in report:  
        if word in sentence:
            count += 1
    return np.log(len(report)/count)        

def sentence_similarity(report, sent1, sent2, stopwords=None):
    if stopwords is None:
        stopwords = []
    all_words = list(set(sent1 + sent2))
    vector1 = [0] * len(all_words)
    vector2 = [0] * len(all_words)
    for w in sent1:
        if w in stopwords:
            continue
        vector1[all_words.index(w)] += 1
    for w in sent2:
        if w in stopwords:
            continue
        vector2[all_words.index(w)] += 1
    vector3 = list(map(lambda x: x[0]/len(sent1) * tfidf(report, x[1]), zip(vector1, all_words)))
    vector4 = list(map(lambda x: x[0]/len(sent1) * tfidf(report, x[1]), zip(vector2, all_words)))
    return 1 - cosine_distance(vector3, vector4)

def build_similarity_matrix(sentences, stop_words):
    similarity_matrix = np.zeros((len(sentences), len(sentences))) 
    for idx1 in range(len(sentences)):
        for idx2 in range(len(sentences)):
            if idx1 == idx2:
                continue 
            similarity_matrix[idx1][idx2] = sentence_similarity(sentences, sentences[idx1], sentences[idx2], stop_words)
    return similarity_matrix

def generate_summary(text):
# def generate_summary(text, top_n=5):

    stop_words = stopwords.words('russian')
    summarize_text = []

    sentences = read_article(text)

    # вот здесь мож надо поменять 
    top_n = max(1, math.ceil(0.3 * len(sentences))) # можно для коротких текстов (<10 предлож) брать больше предложений, типо 50%, а на оч длинных текстах 10-20%

    sentence_similarity_matrix = build_similarity_matrix(sentences, stop_words)

    sentence_similarity_graph = nx.from_numpy_array(sentence_similarity_matrix)
    scores = nx.pagerank(sentence_similarity_graph)

    ranked_sentence = sorted(((scores[i],s) for i,s in enumerate(sentences)), reverse=True)       
    for i in range(top_n):
        summarize_text.append(" ".join(ranked_sentence[i][1]))

    print("Summarize Text: \n", ". ".join(summarize_text))
    return ". ".join(summarize_text)

file_path = 'verdicts_with_c.csv'
dff = pd.read_csv(file_path, on_bad_lines='warn')
# df = dff.loc[:0]
df = dff[['preamble', 'description', 'sentence']].copy()

# может надо написать цикл перебора этих всех штук 
for i in range (1):
    s = generate_summary(df['description'][i]) + ' ' + generate_summary(df['preamble'][i]) + ' ' + generate_summary(df['sentence'][i])
    df.at[i, 'sum_text'] = s
 
df
# generate_summary(df['description'][0], 7)

Summarize Text: 
 и а.в.м.о. и а.в.м.о. т.2 л.д. у него возник конфликт с а.в.м.о который стал высказывать в его адрес оскорбления и угрозы убийством демонстрируя при этом нож однако в данный конфликт вмешался р.м.а после чего он ушел домой. произошел конфликт с а.в.м.о который стал высказывать в его адрес оскорбления в грубой неприличной форме и угрозы убийством демонстрируя при этом нож в который вмешался редько после чего конфликт был улажен а г.д.р. совершил убийство при следующих обстоятельствах в период с дата до дата г.д.р находясь в состоянии алкогольного опьянения у в ходе конфликта с а.в.м.о возникшего на почве личных неприязненных отношений из-за ранее высказанных а.в.м.о. удары руками по лицу а также один удар ножом в левую боковую поверхность груди один удар ножом в левую подмышечную область а также два удара ножом в область левого бедра что со всей очевидностью указывает на умысел подсудимого направленный на лишение жизни а.в.м.о об этом же свидетельствует и выбор им в ка

,preamble,description,sentence,sum_text
0,Дело <№> (29RS0<№>-58)\nПРИГОВОР\nименем Росси...,УСТАНОВИЛ:\nГ.Д.Р. совершил убийство при следу...,ПРИГОВОРИЛ:\nГ.Д.Р. признать виновным в соверш...,и а.в.м.о. и а.в.м.о. т.2 л.д. у него возник к...
1,К делу № 1-141/2019 г.\nПРИГОВОР\nИменем Росси...,"УСТАНОВИЛ:\nЖмурко Ю.Г. совершил убийство, т.е...",ПРИГОВОРИЛ:\nПризнать Жмурко Юрия Геннадьевича...,NaN
2,ПРИГОВОР\nИМЕНЕМ РОССИЙСКОЙ ФЕДЕРАЦИИ\n25 дека...,УСТАНОВИЛ:\nВердиктом коллегии присяжных засед...,ПРИГОВОРИЛ:\nСкородумова Сергея Юрьевича призн...,NaN
3,Решение по уголовному делу\nИнформация по делу...,УСТАНОВИЛ:\nМатвеев С.В. совершил умышленное п...,ПРИГОВОРИЛ:\nпризнать виновным в совершении пр...,NaN
4,ПРИГОВОР\nИМЕНЕМ РОССИЙСКОЙ ФЕДЕРАЦИИ\nг.Ангар...,У С Т А Н О В И Л:\nНа основании вердикта колл...,ПРИГОВОРИЛ:\nПризнать Ш.С.В. виновным в соверш...,NaN
...,...,...,...,...
95,Дело № 1-589/22 (78 BS 0015-01-2022-00231414)\...,У С Т А Н О В И Л :\nПодсудимый Тараканов В.В....,П Р И Г О В О Р И Л :\nПризнать Тараканова Вла...,NaN
96,Дело ***\nПРИГОВОР\nИМЕНЕМ РОССИЙСКОЙ ФЕДЕРАЦИ...,"УСТАНОВИЛ:\nНемтинов С.В. совершил убийство, т...",ПРИГОВОРИЛ:\nПризнать Немтинова Сергея Викторо...,NaN
97,Дело №\nП Р И Г О В О Р\nИменем Российской Фед...,У С Т А Н О В И Л:\nБаженов Д.В. совершил убий...,П Р И Г О В О Р И Л:\nБаженова Д.В. признать в...,NaN
98,Дело №\nПРИГОВОР\nименем Российской Федерации\...,"УСТАНОВИЛ:\nМеликян Т.А. совершил убийство, то...",ПРИГОВОРИЛ:\nМеликяна Т. А. признать виновным ...,NaN


In [14]:
import pandas as pd
import numpy as np
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from stop_words import get_stop_words
from natasha import Doc, Segmenter, MorphVocab, NewsEmbedding, NewsMorphTagger
import re
from pathlib import Path

nltk_data_dir = Path("/Users/vera/Рабочий стол/nltk_data")
nltk_data_dir.mkdir(parents=True, exist_ok=True)
nltk.data.path.append(str(nltk_data_dir))

try:
    nltk.data.find('tokenizers/punkt/PY3/russian.pickle')
    use_nltk_tokenize = True
except LookupError:
    print("Загружаем punkt...")
    try:
        import ssl
        try:
            _create_unverified_https_context = ssl._create_unverified_context
        except AttributeError:
            pass
        else:
            ssl._create_default_https_context = _create_unverified_https_context
        nltk.download('punkt', download_dir=nltk_data_dir, quiet=True)
        use_nltk_tokenize = True
    except Exception as e:
        print(f"Ошибка при загрузке punkt: {e}. Используем natasha для токенизации.")
        use_nltk_tokenize = False

segmenter = Segmenter()
morph_vocab = MorphVocab()
emb = NewsEmbedding()
morph_tagger = NewsMorphTagger(emb)

stop_words = get_stop_words('russian')

def preprocess_sentence(sentence):
    """Предобработка предложения: лемматизация и очистка"""
    sentence = re.sub(r'[^\w\s]', ' ', sentence).strip()
    if not sentence:
        return ""
    
    doc = Doc(sentence)
    doc.segment(segmenter)
    doc.tag_morph(morph_tagger)
    
    words = []
    for token in doc.tokens:
        token.lemmatize(morph_vocab)
        word = token.lemma.lower()
        if word not in stop_words and word.isalpha() and len(word) > 2:
            words.append(word)
    
    return ' '.join(words)

def simple_sent_tokenize(text):
    """Простое разбиение текста на предложения по точкам"""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]

def read_text(text):
    """Разбиение текста на предложения и их предобработка"""
    if not isinstance(text, str) or not text.strip():
        print(f"Пустой или некорректный текст: {text}")
        return [], []
    
    sentences = []
    
    if use_nltk_tokenize:
        try:
            sentences = nltk.sent_tokenize(text, language='russian')
        except LookupError:
            print("NLTK токенизация не удалась, пробуем natasha.")
    
    if not sentences:
        try:
            doc = Doc(text)
            doc.segment(segmenter)
            if doc.sents:
                sentences = [s.text for s in doc.sents]
            else:
                print("Natasha вернула пустую сегментацию, используем простое разбиение.")
        except (AttributeError, ValueError) as e:
            print(f"Natasha не смогла обработать текст: {e}. Используем простое разбиение.")
    
    if not sentences:
        sentences = simple_sent_tokenize(text)
    
    processed_sentences = [preprocess_sentence(sentence) for sentence in sentences]
    valid_indices = [i for i, p in enumerate(processed_sentences) if p and isinstance(p, str) and p.strip()]
    
    return (
        [processed_sentences[i] for i in valid_indices],
        [sentences[i] for i in valid_indices]
    )

def generate_summary(text, top_n=None):
    """Суммаризация текста"""
    if not isinstance(text, str) or not text.strip():
        return "Текст пустой или некорректный"
    
    processed_sentences, original_sentences = read_text(text)
    if not processed_sentences:
        return "Не удалось обработать текст"
    
    num_sentences = len(original_sentences)
    if top_n is None:
        if num_sentences < 10:
            top_n = max(1, int(0.5 * num_sentences))  # 50% для коротких текстов
        else:
            top_n = max(1, int(0.2 * num_sentences))  # 20% для длинных текстов
        # top_n = min(top_n, 5)  # Максимум 5 предложений
    
    if num_sentences < top_n:
        return ' '.join(original_sentences)
    
    vectorizer = TfidfVectorizer()
    try:
        tfidf_matrix = vectorizer.fit_transform(processed_sentences)
    except ValueError:
        return "Ошибка при обработке текста"
    
    similarity_matrix = cosine_similarity(tfidf_matrix)
    
    nx_graph = nx.from_numpy_array(similarity_matrix)
    try:
        scores = nx.pagerank(nx_graph, max_iter=1000)
    except nx.PowerIterationFailedConvergence:
        return "Ошибка при ранжировании предложений"
    
    ranked_sentences = sorted(
        ((scores[i], s, i) for i, s in enumerate(original_sentences)),
        reverse=True
    )
    top_sentences = sorted(
        ranked_sentences[:top_n],
        key=lambda x: x[2] 
    )
    
    summary = '. '.join([sentence for _, sentence, _ in top_sentences]).strip()
    if summary and not summary.endswith('.'):
        summary += '.'
    
    return summary

def summarize_dataframe(dff, top_n=None):
    dff['sum_text'] = dff['description'].apply(lambda x: generate_summary(x, top_n))
    
    text_columns=['preamble', 'sentence', 'sum_text']
    for col in text_columns:
        dff[col] = dff[col].astype(str).replace('nan', '')
    
    dff['text'] = dff[text_columns].agg(' '.join, axis=1)
    
    # df['sum_text'] = df['text'].apply(lambda x: generate_summary(x, top_n))
# def summarize_dataframe(dff, top_n=None):
    # dff['sum_text'] = dff['description'].apply(lambda x: generate_summary(x, top_n))
    return dff

if __name__ == "__main__":
    file_path = 'verdicts_with_c.csv'
    df = pd.read_csv(file_path, on_bad_lines='warn')
    dff = df[['id', 'preamble', 'description', 'sentence']].copy()
    
    # print("Проверка входных данных:")
    # for idx, row in df.iterrows():
    #     if not isinstance(row['preamble'], str) or not isinstance(row['description'], str) or not isinstance(row['sentence'], str):
    #         print(f"Проблема в строке {idx}: preamble={type(row['preamble'])}, description={type(row['description'])}, sentence={type(row['sentence'])}")
    
    # df = summarize_dataframe(df, text_columns=['preamble', 'description', 'sentence'], top_n=2)
    dff = summarize_dataframe(dff, top_n=2)
    
    dff.to_csv('verdicts_with_summaries2.csv', index=False)
    
    for _, row in dff.head().iterrows():
        print(f"\nИндекс: {row.name}")
        print(f"Суммаризация: {row['sum_text']}\n")
        print(f"Итог: {row['text']}")

NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробуем natasha.
NLTK токенизация не удалась, пробу